# LUT Grid Design

Design of the grid of the LUT **without generating a LUT**: only the cost (`cost_function.py`) and the MuJoCo model (single-point methods of `LutGenerator`).

The LUT is interpolated linearly along every axis, so the error of a grid is the error of the straight lines between its nodes. With multilinear interpolation the error is (to first order) a sum of one term per axis, $\sum_d c_d h_d^2$: at the minimum number of points every axis contributes the same error. So **one tolerance for every axis**.

- **W**: nodes that minimize the largest error of the rest activation $u^*(W)$ (minimax)
- **angle, velocity, torque**: exact activations along the axis on a fine grid, on a set of slices (the other inputs fixed), then the points needed to stay within the tolerance

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
import time
import numpy as np

from src.lut.lut import Lut
from src.lut.lut_generator import LutGenerator
from src.lut import grid_design as gd

In [ ]:
# ----- Problem -----
MYO_SIM_MODEL_PATH = os.path.join("assets", "myo_sim", "elbow", "myoelbow_1dof6muscles.xml")
P_VALUE = 3
MAX_THETA_DOT = 600.0 # deg/s, symmetric range [-MAX_THETA_DOT, MAX_THETA_DOT]

# ----- Tolerance on u: the same for every axis -----
METRIC = "rmse" # "max": worst point of all the slices; "rmse": average error
EPS = {"max": 0.01, "rmse": 0.0005}

# ----- Plots -----
SAVE_PLOTS_DIR = None       # e.g. "models/luts/W_all/p3/grid_design_plots"


def plot_path(name: str):
    return os.path.join(SAVE_PLOTS_DIR, name) if SAVE_PLOTS_DIR else None


# the generator is used only for the model: tau_0, K and the solver at single points
gen = LutGenerator(path_model=MYO_SIM_MODEL_PATH, p_value=P_VALUE, w_values=[0.0])
gen.populate_theta(2)                               # the two ends of the joint range of the model
THETA_MIN, THETA_MAX = float(gen.theta[0]), float(gen.theta[-1])

print(f"Muscles     : {gen.muscles}")
print(f"Joint range : [{THETA_MIN:g}, {THETA_MAX:g}] deg")
print(f"Velocity    : [{-MAX_THETA_DOT:g}, {MAX_THETA_DOT:g}] deg/s")
print(f"Tolerance   : {METRIC} error on u <= {EPS[METRIC]:g}")

## Cost Function

$J_p(u) = \sum_i J_i(u_i)$, with $J_i(u) = (1-W)\,u^p + W\,u\ln u$. The minimum of $J_i$ without torque constraint is the rest activation $u^*(W)$, the same for every muscle.

In [ ]:
CONSIDERED_W = 0.2
COMPARE_P = [3]          # p values compared at CONSIDERED_W

gd.plot_cost_terms(P_VALUE, CONSIDERED_W, save_path=plot_path(f"cost_W_{CONSIDERED_W:g}.png"))
Lut.plot_cost_curves([(P_VALUE, W) for W in np.linspace(0.0, 1.0, 11)],
                     title=rf"$J_i(u)$, p = {P_VALUE:g}, W from 0 to 1",
                     save_path=plot_path("cost_all_w.png"))
Lut.plot_cost_curves([(p, CONSIDERED_W) for p in COMPARE_P],
                     save_path=plot_path(f"cost_p_compare_W_{CONSIDERED_W:g}.png"))

## W Grid

$u^*(W)$ has infinite slope at $W = 0$: a uniform grid misses its fast change near 0. The nodes are chosen so that the largest interpolation error of $u^*$ is as small as possible (all the segments with the same error), with segments no longer than `H_MAX` to follow also the dependence of the activations on W through the torque constraint.

In [ ]:
N_SEGMENTS = 14 # number of W intervals (nodes = N_SEGMENTS + 1)
H_MAX = 0.1 # max length of a segment (1.0 = no limit)

w_nodes, w_max_error, w_errors = gd.w_nodes_minimax(N_SEGMENTS, P_VALUE, H_MAX)
W_VALUES = [float(f"{w:.6g}") for w in w_nodes]

print(f"p = {P_VALUE:g}, {len(w_nodes) - 1} segments, h_max = {H_MAX:g}: "
      f"max interpolation error of u* = {w_errors.max():.4f}\n")
print(f"{'segment':>20} | {'W of max error':>14} | {'max error':>9}")
for a, b, wm, e in zip(w_nodes[:-1], w_nodes[1:], w_max_error, w_errors):
    print(f"[{a:.5f}, {b:.5f}] | {wm:14.5f} | {e:9.4f}")

w_fine, u, u_lin = gd.u_rest_error(np.linspace(0.0, 1.0, 11), P_VALUE)
print(f"\nUniform grid of 11 W, for comparison: max error of u* = {np.max(u - u_lin):.4f}")
print(f"\nW_VALUES = {W_VALUES}")

In [ ]:
gd.plot_u_rest(W_VALUES, P_VALUE, save_path=plot_path("u_rest.png"))
gd.plot_u_rest(W_VALUES, P_VALUE, error=True, save_path=plot_path("u_rest_error.png"))

In [ ]:
def representative(values, n: int = 5) -> list:
    # n values of a sorted list, evenly spaced by index: with the W nodes, more of them near 0
    idx = np.unique(np.round(np.linspace(0, len(values) - 1, n)).astype(int))
    return [float(values[i]) for i in idx]


# W of the slices for the other axes: nodes of the W grid (the LUT interpolates the other axes at these W)
SLICE_W = representative(W_VALUES)
# validation W: where the interpolation of u* is worst
VALIDATION_W = [float(f"{w:.3g}") for w in representative(w_max_error, 6)]
print(f"SLICE_W      = {SLICE_W}")
print(f"VALIDATION_W = {VALIDATION_W}")

## Angle Grid

The angle enters the problem only through the torque constraint, $\tau_0(\theta) + K(\theta, \dot\theta)\cdot u = \tau$: the function to look at is the **constrained** solution $u(\theta)$. The slices fix W, velocity and **torque_norm** (not the torque in Nm), because the LUT interpolates along the angle at fixed nodes of (W, velocity, torque_norm).

In [ ]:
SLICE_VELOCITY = [-MAX_THETA_DOT, 0.0, MAX_THETA_DOT]
SLICE_TAU_NORM = [0.05, 0.25, 0.5, 0.75, 0.95] # no edges: there u is analytic
N_FINE_ANGLE = 1301 # 0.1 deg on 130 deg

theta_fine = np.linspace(THETA_MIN, THETA_MAX, N_FINE_ANGLE)
start = time.time()
U_angle, labels_angle = gd.solve_slices(gen, "angle", theta_fine, SLICE_W, None, SLICE_VELOCITY, SLICE_TAU_NORM)
print(f"{U_angle.shape[0]} slices x {N_FINE_ANGLE} angles solved in {time.time() - start:.1f} s")

In [ ]:
for metric in ("max", "rmse"):
    n, error = gd.uniform_points_needed(theta_fine, U_angle, EPS[metric], metric=metric)
    print(f"{metric:>4} <= {EPS[metric]:g}: {n} uniform angles "
          f"(step {(THETA_MAX - THETA_MIN) / (n - 1):.2f} deg)" if n else f"{metric}: more than 501 angles")
ANGLE_POINTS, _ = gd.uniform_points_needed(theta_fine, U_angle, EPS[METRIC], metric=METRIC)

# free nodes: where the angle needs more points
free_nodes = gd.nodes_minimax(theta_fine, U_angle, EPS["max"])
print(f"\nFree nodes with max error <= {EPS['max']:g}: {len(free_nodes)}, "
      f"step from {np.diff(free_nodes).min():.2f} to {np.diff(free_nodes).max():.2f} deg")
print(np.round(free_nodes, 1))

error_now = gd.grid_error(theta_fine, U_angle, np.linspace(THETA_MIN, THETA_MAX, 39))
print(f"\nCurrent grid (39 angles): max {error_now['max']:.4f}, RMSE {error_now['rmse']:.5f}")
print(f"ANGLE_POINTS = {ANGLE_POINTS}")

ANGLE_SPLIT = 100.0         # above it the activations change fast with the angle (joint limit)
uniform = np.linspace(THETA_MIN, THETA_MAX, ANGLE_POINTS)
low = uniform[uniform <= ANGLE_SPLIT]                      # current step up to the split


def angle_nodes(step_high: float) -> np.ndarray:
    n_high = int(np.ceil((THETA_MAX - low[-1]) / step_high))
    return np.r_[low, np.linspace(low[-1], THETA_MAX, n_high + 1)[1:]]


below = theta_fine <= ANGLE_SPLIT
for step in (3.42, 2.5, 2.0, 1.5, 1.25, 1.0):
    e = gd.grid_error(theta_fine, U_angle, angle_nodes(step))
    print(f"step above {ANGLE_SPLIT:g} deg = {step:4.2f}: {len(angle_nodes(step))} angles | "
          f"max below {e['max_along_axis'][below].max():.4f}, above {e['max_along_axis'][~below].max():.4f} | "
          f"RMSE {e['rmse']:.5f}")

ANGLE_STEP_HIGH = 1.5       # from the table above
ANGLE_VALUES = angle_nodes(ANGLE_STEP_HIGH)
print(f"{len(ANGLE_VALUES)} angles: step {np.diff(ANGLE_VALUES)[0]:.2f} deg up to {low[-1]:.1f}, "
      f"then {np.diff(ANGLE_VALUES)[-1]:.2f} deg")

In [ ]:
gd.plot_axis_error(theta_fine, U_angle, np.linspace(THETA_MIN, THETA_MAX, ANGLE_POINTS), "angle [deg]",
                   save_path=plot_path("angle_error.png"))

## Velocity Grid

Same method along the velocity. The number of points must be **odd**, so that velocity 0 is a node.

In [ ]:
SLICE_ANGLE = list(np.linspace(THETA_MIN, THETA_MAX, 5))
SLICE_TAU_NORM = [0.05, 0.25, 0.5, 0.75, 0.95]
N_FINE_VELOCITY = 1201                             # odd: 0 is a point of the fine grid

velocity_fine = np.linspace(-MAX_THETA_DOT, MAX_THETA_DOT, N_FINE_VELOCITY)
start = time.time()
U_velocity, labels_velocity = gd.solve_slices(gen, "velocity", velocity_fine, SLICE_W, SLICE_ANGLE, None,
                                              SLICE_TAU_NORM)
print(f"{U_velocity.shape[0]} slices x {N_FINE_VELOCITY} velocities solved in {time.time() - start:.1f} s")

In [ ]:
for metric in ("max", "rmse"):
    n, error = gd.uniform_points_needed(velocity_fine, U_velocity, EPS[metric], metric=metric, odd=True)
    print(f"{metric:>4} <= {EPS[metric]:g}: {n} uniform velocities "
          f"(step {2 * MAX_THETA_DOT / (n - 1):.1f} deg/s)" if n else f"{metric}: more than 501 velocities")
VELOCITY_POINTS, _ = gd.uniform_points_needed(velocity_fine, U_velocity, EPS[METRIC], metric=METRIC, odd=True)
VELOCITY_POINTS = 23    # knee of the convergence curve: max error 0.024 (11 points) -> 0.0056 (23 points)

free_nodes = gd.nodes_minimax(velocity_fine, U_velocity, EPS["max"])
print(f"\nFree nodes with max error <= {EPS['max']:g}: {len(free_nodes)}, "
      f"step from {np.diff(free_nodes).min():.1f} to {np.diff(free_nodes).max():.1f} deg/s")
print(np.round(free_nodes, 1))
print(f"\nVELOCITY_POINTS = {VELOCITY_POINTS}")

In [ ]:
gd.plot_axis_error(velocity_fine, U_velocity, np.linspace(-MAX_THETA_DOT, MAX_THETA_DOT, VELOCITY_POINTS),
                   "velocity [deg/s]", save_path=plot_path("velocity_error.png"))

## Torque Grid

Same method along torque_norm, with the nodes of `LutGenerator.populate_tau_norm`: uniform $s$ moved towards 0 and 1 by `edge_density`, $\tau_{norm} = s - e\,\sin(2\pi s)/(2\pi)$. For every `edge_density` the smallest number of points within the tolerance; the best pair goes in the dataset.

In [ ]:
SLICE_ANGLE = list(np.linspace(THETA_MIN, THETA_MAX, 5))
SLICE_VELOCITY = [-MAX_THETA_DOT, 0.0, MAX_THETA_DOT]
EDGE_DENSITIES = (0.0, 0.25, 0.5, 0.75, 0.8, 0.85)
N_FINE_TORQUE = 2001

tau_norm_fine = np.linspace(0.0, 1.0, N_FINE_TORQUE)
start = time.time()
U_torque, labels_torque = gd.solve_slices(gen, "torque_norm", tau_norm_fine, SLICE_W, SLICE_ANGLE,
                                          SLICE_VELOCITY, None)
print(f"{U_torque.shape[0]} slices x {N_FINE_TORQUE} torques solved in {time.time() - start:.1f} s")

In [ ]:
for metric in ("max", "rmse"):
    needed = gd.torque_points_needed(tau_norm_fine, U_torque, EPS[metric], EDGE_DENSITIES, metric=metric)
    print(f"{metric} <= {EPS[metric]:g}: " +
          ", ".join(f"edge {e:.2f} -> {n if n else '> 501'}" for e, (n, _) in needed.items()))
    
for e, (n, error) in needed.items():
    print(f"  edge_density {e:.2f}: {n if n else '> 501'} points")

found = {e: n for e, (n, _) in needed.items() if n is not None}
EDGE_DENSITY = min(found, key=found.get)
TORQUE_POINTS = found[EDGE_DENSITY]
error_now = gd.grid_error(tau_norm_fine, U_torque, gd.tau_norm_nodes(61, 0.5))
print(f"\nCurrent grid (61 points, edge_density 0.5): max {error_now['max']:.4f}, RMSE {error_now['rmse']:.5f}")
print(f"TORQUE_POINTS = {TORQUE_POINTS}, EDGE_DENSITY = {EDGE_DENSITY}")

In [ ]:
gd.plot_axis_error(tau_norm_fine, U_torque, gd.tau_norm_nodes(TORQUE_POINTS, EDGE_DENSITY), "torque_norm",
                   save_path=plot_path("torque_error.png"))

In [ ]:
from src.utils.plotting import plot_cartesian

# interpolation error against the number of points, for every axis (uniform grids, torque with EDGE_DENSITY)
axes = {
    "angle": (theta_fine, U_angle, range(9, 131, 4),
              lambda n: np.linspace(THETA_MIN, THETA_MAX, n)),
    "velocity": (velocity_fine, U_velocity, range(5, 62, 2),             # odd: 0 is a node
                 lambda n: np.linspace(-MAX_THETA_DOT, MAX_THETA_DOT, n)),
    "torque_norm": (tau_norm_fine, U_torque, range(10, 201, 5),
                    lambda n: gd.tau_norm_nodes(n, EDGE_DENSITY)),
}
for name, (x_fine, U, counts, nodes_of) in axes.items():
    counts = np.array(list(counts))
    errors = [gd.grid_error(x_fine, U, nodes_of(n)) for n in counts]
    plot_cartesian(title=f"Interpolation error of u against the number of points: {name}",
                   x=counts, y=[np.array([e["max"] for e in errors]), np.array([e["rmse"] for e in errors])],
                   labels=["max", "RMSE"], x_label="points", y_label="error on u", log_y=True,
                   save_path=plot_path(f"convergence_{name}.png"))

## Validation Dataset

In [ ]:
def midpoints(nodes, every: int = 1) -> np.ndarray:
    """Midpoints of the cells of a training grid, one every `every` cells (always the first and the last):
    there the linear interpolation between the nodes is worst."""
    nodes = np.asarray(nodes, dtype=float)
    mid = 0.5 * (nodes[:-1] + nodes[1:])
    idx = np.unique(np.r_[np.arange(0, len(mid), every), len(mid) - 1])
    return mid[idx]


mid_angles = 0.5 * (ANGLE_VALUES[:-1] + ANGLE_VALUES[1:])
VALIDATION_ANGLES = np.r_[midpoints(ANGLE_VALUES[ANGLE_VALUES <= low[-1]], every=3),
                          midpoints(ANGLE_VALUES[ANGLE_VALUES >= low[-1]], every=2)]
VALIDATION_VELOCITIES = midpoints(np.linspace(-MAX_THETA_DOT, MAX_THETA_DOT, VELOCITY_POINTS), every=2)
VALIDATION_TAU_NORM = midpoints(gd.tau_norm_nodes(TORQUE_POINTS, EDGE_DENSITY), every=3)

n_validation = (len(VALIDATION_W) * len(VALIDATION_ANGLES) * len(VALIDATION_VELOCITIES)
                * len(VALIDATION_TAU_NORM))                     # no analytic edges: every point is solved
print(f"Validation: {len(VALIDATION_W)} W x {len(VALIDATION_ANGLES)} angles x "
      f"{len(VALIDATION_VELOCITIES)} velocities x {len(VALIDATION_TAU_NORM)} torques = {n_validation:,} points")

## Dataset Parameters

The parameters for `dataset_generation.py`. The edges of the torque (torque_norm 0 and 1) are analytic, so the points to solve are $n_W \cdot n_\theta \cdot n_{\dot\theta} \cdot (n_\tau - 2)$.

In [ ]:
SAVE_PARAMS_PATH = "models/luts/W_all/p3/grid_params.json"

params = {
    "P_VALUE": P_VALUE,
    "W_VALUES": W_VALUES,
    "ANGLE_POINTS": ANGLE_VALUES.round(3).tolist(),      # explicit angles: denser above 100 deg
    "MAX_THETA_DOT": MAX_THETA_DOT,
    "VELOCITY_POINTS": int(VELOCITY_POINTS),
    "TORQUE_POINTS": int(TORQUE_POINTS),
    "EDGE_DENSITY": float(EDGE_DENSITY),
    "tolerance": {"metric": METRIC, "eps": EPS[METRIC]},
    "VALIDATION_W": VALIDATION_W,
    "VALIDATION_ANGLES": VALIDATION_ANGLES.round(4).tolist(),
    "VALIDATION_VELOCITIES": VALIDATION_VELOCITIES.round(4).tolist(),
    "VALIDATION_TAU_NORM": VALIDATION_TAU_NORM.round(6).tolist(),
}

n_points = len(W_VALUES) * len(ANGLE_VALUES) * VELOCITY_POINTS * (TORQUE_POINTS - 2)
n_points_now = 11 * 37 * 41 * (61 - 2)

for name, value in params.items():
    print(f"{name:<16}: {value}")
print(f"\nPoints to solve : {n_points:,} (current LUT: {n_points_now:,}, ratio {n_points / n_points_now:.2f})")

if SAVE_PARAMS_PATH:
    os.makedirs(os.path.dirname(SAVE_PARAMS_PATH), exist_ok=True)
    with open(SAVE_PARAMS_PATH, "w") as f:
        json.dump(params, f, indent=4)
    print(f"Saved in {SAVE_PARAMS_PATH}")